# Word2Vec using Gensim

In [ ]:
import subprocess
import sys
import os
import csv
import json

# ================================================================
# INSTALL LIBRARIES
# ================================================================
def maintain_dependencies():
    required_libraries = ['numpy', 'scipy', 'gensim', 'nltk']
    for lib in required_libraries:
        try:
            __import__(lib)
        except ImportError:
            print(f"Installing {lib}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

maintain_dependencies()

# ================================================================
# IMPORTS
# ================================================================
import numpy as np
from scipy.stats import spearmanr, pearsonr
from gensim.models import Word2Vec  # ← changed from FastText

WORDNET_AVAILABLE = False
try:
    import nltk
    from nltk.corpus import wordnet
    try:
        wordnet.synsets("run")
        WORDNET_AVAILABLE = True
        print("WordNet loaded successfully.")
    except LookupError:
        print("WordNet not found locally. Attempting download...")
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        try:
            wordnet.synsets("run")
            WORDNET_AVAILABLE = True
            print("WordNet downloaded and loaded.")
        except Exception:
            print("WARNING: WordNet unavailable. Cross-lingual validation will be skipped.")
except Exception:
    print("WARNING: NLTK unavailable. Cross-lingual validation will be skipped.")


# ================================================================
# ISIZULU → ENGLISH BILINGUAL DICTIONARY  (loaded from file)
# ================================================================
DICT_FILE = "isizulu_english_dict.json"

def load_bilingual_dict(filepath):
    """Load the isiZulu→English bilingual dictionary from a JSON file."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Bilingual dictionary loaded : {len(data)} entries  ({filepath})")
        return data
    except FileNotFoundError:
        print(f"ERROR: Dictionary file '{filepath}' not found.")
        print("       Place 'isizulu_english_dict.json' in the same directory as this script.")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"ERROR: Could not parse '{filepath}': {e}")
        sys.exit(1)

ISIZULU_ENGLISH = load_bilingual_dict(DICT_FILE)


# ================================================================
# ISIZULU NOUN CLASS FILTER
# ================================================================
# isiZulu has 15 noun classes, each with characteristic prefixes.
# True synonyms must share the same noun class (same prefix group).
# This filter discards candidates whose prefix signals a different
# grammatical class from the target word.
#
# Prefix table (singular → plural pairs where applicable):
#   Class 1/2  : um-/umu-  → aba-/abe-     (people)
#   Class 3/4  : um-/umu-  → imi-           (things/abstract)
#   Class 5/6  : i-/ili-   → ama-           (various)
#   Class 7/8  : is-/isi-  → iz-/izi-       (things/languages)
#   Class 9/10 : in-/im-   → izin-/izim-    (animals/things)
#   Class 11   : ul-/ulu-  → izin-/izim-    (abstract/long things)
#   Class 14   : ubu-/ub-                   (abstract nouns)
#   Class 15   : uku-/uk-                   (infinitives/verbal nouns)
#   Class 17   : ku-                        (locative)
#
# Words in the same PAIR (e.g. class 1 & 2) are treated as the same
# group because singular and plural forms are legitimate matches.

NOUN_CLASS_GROUPS = [
    # Group 0 : Class 1 & 2  — people (um/umu → aba/abe)
    {"um", "umu", "aba", "abe", "ob"},
    # Group 1 : Class 3 & 4  — things/abstract (um/umu → imi)
    # NOTE: um/umu overlaps with class 1; noun class detection is
    # approximate without a full morphological analyser.
    {"imi"},
    # Group 2 : Class 5 & 6  — general (i/ili → ama)
    {"i", "ili", "ama", "ame"},
    # Group 3 : Class 7 & 8  — things/languages (isi/is → izi/iz)
    {"isi", "is", "izi", "iz"},
    # Group 4 : Class 9 & 10 — animals/things (in/im → izin/izim)
    {"in", "im", "izin", "izim"},
    # Group 5 : Class 11     — abstract/long (ulu/ul)
    {"ulu", "ul"},
    # Group 6 : Class 14     — abstract nouns (ubu/ub)
    {"ubu", "ub"},
    # Group 7 : Class 15     — infinitives (uku/uk)
    {"uku", "uk"},
    # Group 8 : Class 17     — locative (ku)
    {"ku"},
]

# Ordered longest-first so "uku" is matched before "u", etc.
_ALL_PREFIXES = sorted(
    {p for group in NOUN_CLASS_GROUPS for p in group},
    key=len, reverse=True
)

def get_noun_class_group(word: str) -> int:
    """
    Return the index of the noun-class group for `word`, or -1 if unknown.
    Matches the longest known prefix at the start of the word.
    """
    w = word.lower()
    for prefix in _ALL_PREFIXES:
        if w.startswith(prefix):
            for idx, group in enumerate(NOUN_CLASS_GROUPS):
                if prefix in group:
                    return idx
    return -1  # unknown / not matched


def same_noun_class(word1: str, word2: str) -> bool:
    """
    Return True if both words belong to the same noun-class group.
    If either word's class is unknown (-1), we allow it through
    (benefit of the doubt) rather than discarding it.
    """
    g1 = get_noun_class_group(word1)
    g2 = get_noun_class_group(word2)
    if g1 == -1 or g2 == -1:
        return True   # can't determine — keep candidate
    return g1 == g2


# ================================================================
# COSINE SIMILARITY
# ================================================================
def cosine_similarity(vec1, vec2):
    dot   = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)


# ================================================================
# LOAD CORPUS
# ================================================================
def load_text_file(filepath):
    sentences = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                tokens = line.lower().strip().split()
                if tokens:
                    sentences.append(tokens)
    except Exception as e:
        print("Error loading corpus:", e)
    return sentences


# ================================================================
# LOAD TEST PAIRS
# ================================================================
def load_test_pairs(filepath):
    pairs = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader)
            print("CSV columns detected:", header)
            for row in reader:
                if len(row) < 6:
                    continue
                w1 = row[0].strip().strip('"').strip(',')
                w2 = row[1].strip().strip('"').strip(',')
                try:
                    score = float(row[5].strip())
                except ValueError:
                    continue
                pairs.append((w1, w2, score))
        print(f"Loaded {len(pairs)} pairs")
    except Exception as e:
        print("Error loading test pairs:", e)
    return pairs


# ================================================================
# CLASSIFICATION METRICS
# ================================================================
def confusion_matrix_np(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tn, fp, fn, tp

def accuracy_np(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    return (tp + tn) / total if total > 0 else 0

def precision_np(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0

def recall_np(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def f1_np(p, r):
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0


# ================================================================
# SYNONYM CANDIDATE GENERATION  (with noun class filter)
# ================================================================
def generate_synonym_candidates(model, target_words=None, topn=10, min_similarity=0.3):
    """
    Generate synonym candidates using Word2Vec nearest neighbours,
    then discard any candidate that belongs to a different isiZulu
    noun class than the query word.

    NOTE: Unlike FastText, Word2Vec cannot generate vectors for
    out-of-vocabulary words — words must appear in training data.
    """
    if target_words is None:
        target_words = list(model.wv.key_to_index.keys())

    candidates = []
    total_discarded = 0

    for word in target_words:
        if word not in model.wv.key_to_index:
            # Word2Vec has no subword fallback — skip OOV words entirely
            candidates.append({
                "query_word":        word,
                "candidate":         "N/A",
                "cosine_similarity": 0.0,
                "rank":              0,
                "above_threshold":   "NO",
                "noun_class_match":  "N/A",
                "note":              "Word not found in model vocabulary (Word2Vec has no OOV fallback)"
            })
            continue

        try:
            similar = model.wv.most_similar(word, topn=topn * 2)
            kept_rank = 0
            for candidate, score in similar:
                if score < min_similarity:
                    continue

                # ── NOUN CLASS FILTER ──────────────────────────────────
                if not same_noun_class(word, candidate):
                    total_discarded += 1
                    continue
                # ──────────────────────────────────────────────────────

                kept_rank += 1
                candidates.append({
                    "query_word":        word,
                    "candidate":         candidate,
                    "cosine_similarity": round(float(score), 4),
                    "rank":              kept_rank,
                    "above_threshold":   "YES" if score >= min_similarity else "NO",
                    "noun_class_match":  "YES",
                    "note": (
                        "Strong candidate"   if score >= 0.6 else
                        "Moderate candidate" if score >= 0.4 else
                        "Weak candidate"     if score >= min_similarity else
                        "Below threshold — likely unrelated"
                    )
                })
                if kept_rank >= topn:
                    break

        except KeyError:
            candidates.append({
                "query_word":        word,
                "candidate":         "N/A",
                "cosine_similarity": 0.0,
                "rank":              0,
                "above_threshold":   "NO",
                "noun_class_match":  "N/A",
                "note":              "Word not found in model vocabulary"
            })

    print(f"Noun-class filter discarded : {total_discarded} candidates")
    return candidates


# ================================================================
# WORDNET CROSS-LINGUAL VALIDATION
# ================================================================
def wordnet_validate(query_zulu, candidate_zulu, bilingual_dict, wn):
    result = {
        "wordnet_validated": None,
        "wordnet_score":     None,
        "shared_synset":     None,
        "query_english":     None,
        "candidate_english": None,
    }

    q_eng = bilingual_dict.get(query_zulu.lower())
    c_eng = bilingual_dict.get(candidate_zulu.lower())

    if not q_eng or not c_eng:
        return result

    result["query_english"]     = ", ".join(q_eng)
    result["candidate_english"] = ", ".join(c_eng)

    q_synsets = []
    for word in q_eng:
        q_synsets.extend(wn.synsets(word.replace(" ", "_")))

    c_synsets = []
    for word in c_eng:
        c_synsets.extend(wn.synsets(word.replace(" ", "_")))

    if not q_synsets or not c_synsets:
        result["wordnet_validated"] = False
        return result

    shared = {s.name() for s in q_synsets} & {s.name() for s in c_synsets}
    if shared:
        result["wordnet_validated"] = True
        result["shared_synset"]     = list(shared)[0]
        result["wordnet_score"]     = 1.0
        return result

    best_score = 0.0
    for qs in q_synsets:
        for cs in c_synsets:
            try:
                sim = qs.wup_similarity(cs)
                if sim and sim > best_score:
                    best_score = sim
            except Exception:
                continue

    result["wordnet_score"]     = round(best_score, 4) if best_score > 0 else None
    result["wordnet_validated"] = best_score >= 0.75
    return result


# ================================================================
# ANNOTATION SCAFFOLD
# ================================================================
def generate_annotation_scaffold(candidates, output_file, min_similarity=0.4, max_pairs=200):
    filtered = [
        c for c in candidates
        if c["cosine_similarity"] >= min_similarity and c["candidate"] != "N/A"
    ]
    filtered.sort(key=lambda x: x["cosine_similarity"], reverse=True)
    filtered = filtered[:max_pairs]

    fieldnames = [
        "query_word", "candidate_synonym", "cosine_similarity", "rank",
        "noun_class_match", "model_note",
        "human_score_0_to_10", "can_substitute_YES_or_NO", "annotator_notes",
    ]

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        f.write("# ISIZULU SYNONYM ANNOTATION TASK\n")
        f.write("# human_score_0_to_10 : 0=unrelated, 10=perfect synonyms\n")
        f.write("# can_substitute      : YES if meaning is preserved when swapped\n")
        f.write("# noun_class_match    : YES = same isiZulu noun class (more reliable)\n")
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for c in filtered:
            writer.writerow({
                "query_word":               c["query_word"],
                "candidate_synonym":        c["candidate"],
                "cosine_similarity":        c["cosine_similarity"],
                "rank":                     c["rank"],
                "noun_class_match":         c.get("noun_class_match", "N/A"),
                "model_note":               c["note"],
                "human_score_0_to_10":      "",
                "can_substitute_YES_or_NO": "",
                "annotator_notes":          "",
            })

    print(f"Annotation scaffold saved → {output_file}  ({len(filtered)} pairs)")
    return len(filtered)


# ================================================================
# INTERACTIVE SYNONYM LOOKUP  (with noun class filter)
# ================================================================
def interactive_synonym_lookup(model):
    print("\n" + "="*60)
    print("  INTERACTIVE SYNONYM LOOKUP")
    print("="*60)
    print("  Enter an isiZulu word to retrieve its synonyms.")
    print("  Candidates are filtered by matching noun class.")
    print("  NOTE: Word2Vec cannot look up out-of-vocabulary words.")
    print("  Commands:")
    print("    topn N   — change how many results to show (default 10)")
    print("    quit     — exit lookup mode\n")

    topn = 10

    while True:
        word = input("  Enter word: ").strip().lower()

        if not word:
            continue

        if word == "quit":
            print("  Exiting synonym lookup.\n")
            break

        if word.startswith("topn "):
            try:
                topn = int(word.split()[1])
                print(f"  Results per query set to {topn}.")
            except (ValueError, IndexError):
                print("  Usage: topn N  (e.g. topn 5)")
            continue

        print()

        # ── OOV check (Word2Vec has no subword fallback) ──────────────
        if word not in model.wv.key_to_index:
            print(f"  '{word}' is not in the Word2Vec vocabulary.")
            print("  Word2Vec requires a word to appear in the training corpus.")
            print("  (FastText could still embed it via character n-grams.)")
            print()
            continue
        # ──────────────────────────────────────────────────────────────

        # Show the detected noun class group for the query word
        nc_group = get_noun_class_group(word)
        if nc_group != -1:
            group_prefixes = ", ".join(sorted(NOUN_CLASS_GROUPS[nc_group]))
            print(f"  Detected noun class group : {nc_group}  (prefixes: {group_prefixes})")
        else:
            print(f"  Noun class : unknown — noun class filter will allow all candidates")

        # --- Word2Vec nearest neighbours with noun class filter ---
        try:
            similar_raw = model.wv.most_similar(word, topn=topn * 2)

            # Apply noun class filter
            similar_filtered  = []
            similar_discarded = []
            for candidate, score in similar_raw:
                if same_noun_class(word, candidate):
                    similar_filtered.append((candidate, score))
                else:
                    similar_discarded.append((candidate, score))

            similar_filtered = similar_filtered[:topn]

            print(f"\n  Word2Vec synonym candidates for '{word}'")
            print(f"  (noun-class filtered — {len(similar_discarded)} candidates removed):")
            print(f"  {'#':<4} {'Word':<25} {'Cosine':>8}  Label")
            print("  " + "-"*55)

            if similar_filtered:
                for rank, (candidate, score) in enumerate(similar_filtered, 1):
                    label = (
                        "Strong"   if score >= 0.6 else
                        "Moderate" if score >= 0.4 else
                        "Weak"
                    )
                    print(f"  {rank:<4} {candidate:<25} {score:>8.4f}  {label}")
            else:
                print("  No candidates survived the noun class filter.")
                print("  (Try lowering the threshold or checking the word's prefix.)")

            if similar_discarded:
                print(f"\n  Discarded (different noun class) — top 5:")
                for candidate, score in similar_discarded[:5]:
                    g = get_noun_class_group(candidate)
                    print(f"    {candidate:<25} {score:.4f}  (group {g})")

        except KeyError:
            print(f"  '{word}' not found in Word2Vec vocabulary.")

        # --- Bilingual dictionary + WordNet ---
        english_translations = ISIZULU_ENGLISH.get(word)
        if english_translations:
            print(f"\n  English translation(s): {', '.join(english_translations)}")

            if WORDNET_AVAILABLE:
                from nltk.corpus import wordnet as wn_corpus
                wn_synonyms = set()
                for eng in english_translations:
                    for synset in wn_corpus.synsets(eng.replace(" ", "_")):
                        for lemma in synset.lemmas():
                            syn = lemma.name().replace("_", " ")
                            if syn.lower() not in [e.lower() for e in english_translations]:
                                wn_synonyms.add(syn)

                if wn_synonyms:
                    print(f"\n  WordNet English synonyms (via '{', '.join(english_translations)}'):")
                    for i, syn in enumerate(sorted(wn_synonyms), 1):
                        print(f"    {i}. {syn}")
                else:
                    print("  No WordNet synonyms found for this translation.")
        else:
            print(f"\n  '{word}' not in bilingual dictionary — WordNet lookup skipped.")
            print("  Add it to 'isizulu_english_dict.json' for cross-lingual validation.")

        print()


# ================================================================
# MAIN
# ================================================================
if __name__ == "__main__":

    CORPUS_FILE      = "isizulu_corpus.txt"
    TEST_FILE        = "human scores_test_pair.csv"
    OUTPUT_EVAL      = "word2vec_results.csv"           # ← renamed from fasttext_results
    OUTPUT_SYNONYMS  = "isizulu_synonym_candidates.csv"
    OUTPUT_VALIDATED = "isizulu_wordnet_validated.csv"
    OUTPUT_ANNOTATION= "isizulu_annotation_scaffold.csv"
    OUTPUT_METRICS   = "evaluation_metrics.txt"

    # ================================================================
    # 1. LOAD CORPUS
    # ================================================================
    print("\n" + "="*60)
    print("STEP 1: Loading corpus")
    print("="*60)
    sentences = load_text_file(CORPUS_FILE)

    if not sentences:
        print("Corpus empty — check that the file exists and has content.")
        sys.exit()

    print(f"Sentences loaded : {len(sentences)}")
    total_tokens = sum(len(s) for s in sentences)
    print(f"Total tokens     : {total_tokens}")

    if total_tokens < 100_000:
        print("WARNING: Corpus is very small. Embeddings may be unreliable.")
        print("WARNING: Word2Vec is more sensitive to corpus size than FastText,")
        print("         as it has no subword fallback for rare/unseen words.")


    # ================================================================
    # 2. TRAIN WORD2VEC
    # ================================================================
    print("\n" + "="*60)
    print("STEP 2: Training Word2Vec model")
    print("="*60)

    import multiprocessing
    cpu_count = multiprocessing.cpu_count()

    model = Word2Vec(               # ← changed from FastText
        sentences=sentences,
        vector_size=100,            # embedding dimensions
        window=4,                   # context window size
        min_count=2,                # ignore words appearing fewer times
        sg=1,                       # 1 = skip-gram, 0 = CBOW
        epochs=5,
        workers=cpu_count,
        alpha=0.05,
        sample=1e-1,
        # min_n, max_n removed — those are FastText-only subword parameters
    )

    print(f"Vocabulary size  : {len(model.wv)}")
    print(f"NOTE: Word2Vec vocabulary is fixed to words seen during training.")
    print(f"      Any word not in this vocab cannot be embedded at inference time.")


    # ================================================================
    # 3. INTERACTIVE SYNONYM LOOKUP
    # ================================================================
    print("\n" + "="*60)
    ans = input("  Would you like to look up synonyms interactively? (yes/no): ").strip().lower()
    if ans in ("yes", "y"):
        interactive_synonym_lookup(model)


    # ================================================================
    # 4. GENERATE SYNONYM CANDIDATES
    # ================================================================
    print("\n" + "="*60)
    print("STEP 3: Generating synonym candidates")
    print("="*60)

    target_words = [w for w in ISIZULU_ENGLISH.keys() if w in model.wv.key_to_index]
    oov_count = len(ISIZULU_ENGLISH) - len(target_words)
    print(f"Target words in vocab : {len(target_words)} / {len(ISIZULU_ENGLISH)}")
    if oov_count > 0:
        print(f"OOV words skipped     : {oov_count}  (Word2Vec cannot embed these)")

    candidates = generate_synonym_candidates(
        model, target_words=target_words, topn=10, min_similarity=0.3,
    )
    print(f"Total candidate pairs generated : {len(candidates)}")

    with open(OUTPUT_SYNONYMS, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            "query_word", "candidate", "cosine_similarity", "rank",
            "above_threshold", "noun_class_match", "note"
        ])
        writer.writeheader()
        writer.writerows(candidates)
    print(f"All synonym candidates saved → {OUTPUT_SYNONYMS}")


    # ================================================================
    # 5. WORDNET CROSS-LINGUAL VALIDATION
    # ================================================================
    print("\n" + "="*60)
    print("STEP 4: WordNet cross-lingual validation")
    print("="*60)

    validated_rows = []
    wn_checked = wn_confirmed = 0

    if WORDNET_AVAILABLE:
        from nltk.corpus import wordnet as wn_corpus
        for c in candidates:
            if c["candidate"] == "N/A" or c["cosine_similarity"] < 0.3:
                continue
            wn_result = wordnet_validate(c["query_word"], c["candidate"], ISIZULU_ENGLISH, wn_corpus)
            wn_checked += 1
            row = {**c, **wn_result}
            if wn_result["wordnet_validated"] is True:
                row["final_verdict"] = "CONFIRMED SYNONYM"
                wn_confirmed += 1
            elif wn_result["wordnet_validated"] is False:
                row["final_verdict"] = "NOT CONFIRMED"
            else:
                row["final_verdict"] = (
                    "STRONG CANDIDATE (no WordNet data)" if c["cosine_similarity"] >= 0.5
                    else "CANDIDATE (no WordNet data)"
                )
            validated_rows.append(row)

        print(f"Pairs checked via WordNet  : {wn_checked}")
        print(f"WordNet-confirmed synonyms : {wn_confirmed}")
    else:
        for c in candidates:
            if c["candidate"] == "N/A":
                continue
            row = {**c,
                   "wordnet_validated": None, "wordnet_score": None,
                   "shared_synset": None,
                   "query_english":     ISIZULU_ENGLISH.get(c["query_word"].lower(), ["unknown"])[0],
                   "candidate_english": ISIZULU_ENGLISH.get(c["candidate"].lower(), ["unknown"])[0],
                   "final_verdict": (
                       "STRONG CANDIDATE"   if c["cosine_similarity"] >= 0.6 else
                       "MODERATE CANDIDATE" if c["cosine_similarity"] >= 0.4 else
                       "WEAK CANDIDATE"
                   )}
            validated_rows.append(row)

    if validated_rows:
        fieldnames = [
            "query_word", "candidate", "cosine_similarity", "rank",
            "above_threshold", "noun_class_match", "note",
            "query_english", "candidate_english",
            "wordnet_validated", "wordnet_score", "shared_synset", "final_verdict",
        ]
        with open(OUTPUT_VALIDATED, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(validated_rows)
        print(f"Validated results saved → {OUTPUT_VALIDATED}")

        confirmed = [r for r in validated_rows if "CONFIRMED" in r.get("final_verdict","") or "STRONG" in r.get("final_verdict","")]
        print(f"\nTop confirmed / strong synonym candidates (sample):")
        print(f"  {'Query':<20} {'Candidate':<20} {'Cosine':>8}  Verdict")
        print("  " + "-"*70)
        for r in confirmed[:15]:
            print(f"  {r['query_word']:<20} {r['candidate']:<20} {r['cosine_similarity']:>8.4f}  {r['final_verdict']}")


    # ================================================================
    # 6. ANNOTATION SCAFFOLD
    # ================================================================
    print("\n" + "="*60)
    print("STEP 5: Generating native speaker annotation scaffold")
    print("="*60)

    n_annotation_pairs = generate_annotation_scaffold(
        candidates=candidates, output_file=OUTPUT_ANNOTATION,
        min_similarity=0.4, max_pairs=200,
    )


    # ================================================================
    # 7. EVALUATE AGAINST HEURISTIC TEST PAIRS
    # ================================================================
    print("\n" + "="*60)
    print("STEP 6: Evaluating against heuristic test pairs")
    print("="*60)

    isi_test_pairs = load_test_pairs(TEST_FILE)

    if not isi_test_pairs:
        print("No test pairs loaded — skipping correlation evaluation.")
    else:
        cosine_scores = []
        human_scores  = []
        results       = []
        skipped       = 0
        skip_reasons  = {}

        print("\nCalculating similarities...")
        for i, (w1, w2, hscore) in enumerate(isi_test_pairs):
            if i % 500 == 0:
                print(f"  Processing {i} / {len(isi_test_pairs)}")
            try:
                # Word2Vec raises KeyError for OOV words (no subword fallback)
                vec1 = model.wv[w1]
                vec2 = model.wv[w2]
                cos  = cosine_similarity(vec1, vec2)
                cosine_scores.append(cos)
                human_scores.append(hscore)
                results.append({"word1": w1, "word2": w2, "human_score": hscore, "cosine_similarity": cos})
            except KeyError:
                # OOV is expected and common with Word2Vec
                skip_reasons["OOV"] = skip_reasons.get("OOV", 0) + 1
                skipped += 1
            except Exception as e:
                reason = type(e).__name__
                skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
                skipped += 1

        print(f"\nPairs used    : {len(cosine_scores)}")
        print(f"Pairs skipped : {skipped}")
        if skip_reasons:
            for reason, count in skip_reasons.items():
                print(f"  Skip reason '{reason}': {count} pairs")

        cosine_std = np.std(cosine_scores)
        human_std  = np.std(human_scores)

        if len(cosine_scores) >= 2 and cosine_std > 0 and human_std > 0:
            rho,  rho_p  = spearmanr(human_scores, cosine_scores)
            pear, pear_p = pearsonr(human_scores, cosine_scores)

            print(f"\nSpearman rho : {rho:.4f}  (p = {rho_p:.4e})")
            print(f"Pearson  r   : {pear:.4f}  (p = {pear_p:.4e})")

            human_median  = np.median(human_scores)
            cosine_median = np.median(cosine_scores)
            y_true = (np.array(human_scores)  >= human_median).astype(int)
            y_pred = (np.array(cosine_scores) >= cosine_median).astype(int)
            tn, fp, fn, tp = confusion_matrix_np(y_true, y_pred)
            accuracy  = accuracy_np(tp, tn, fp, fn)
            precision = precision_np(tp, fp)
            recall    = recall_np(tp, fn)
            f1        = f1_np(precision, recall)

            print(f"\nAccuracy  : {accuracy:.4f}")
            print(f"Precision : {precision:.4f}")
            print(f"Recall    : {recall:.4f}")
            print(f"F1        : {f1:.4f}")
            print(f"\nConfusion Matrix  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

            with open(OUTPUT_EVAL, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=["word1","word2","human_score","cosine_similarity"])
                writer.writeheader()
                writer.writerows(results)
            print(f"\nEvaluation results saved → {OUTPUT_EVAL}")

            with open(OUTPUT_METRICS, 'w') as f:
                f.write("ISIZULU WORD2VEC SYNONYM EVALUATION REPORT\n")
                f.write("="*60 + "\n\n")
                f.write(f"Sentences  : {len(sentences)}\n")
                f.write(f"Tokens     : {total_tokens}\n")
                f.write(f"Vocab size : {len(model.wv)}\n\n")
                f.write(f"Spearman rho : {rho:.4f}  (p = {rho_p:.4e})\n")
                f.write(f"Pearson  r   : {pear:.4f}  (p = {pear_p:.4e})\n\n")
                f.write(f"Accuracy  : {accuracy:.4f}\n")
                f.write(f"Precision : {precision:.4f}\n")
                f.write(f"Recall    : {recall:.4f}\n")
                f.write(f"F1        : {f1:.4f}\n\n")
                f.write(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
            print(f"Metrics saved → {OUTPUT_METRICS}")

        else:
            print("ERROR: Cannot compute correlation — check variance in scores.")


    # ================================================================
    # SUMMARY + offer another lookup session at the end
    # ================================================================
    print("\n" + "="*60)
    print("COMPLETE — Output files produced:")
    print("="*60)
    for fname, desc in [
        (OUTPUT_SYNONYMS,   "All synonym candidates"),
        (OUTPUT_VALIDATED,  "WordNet validated candidates"),
        (OUTPUT_ANNOTATION, "Annotation scaffold"),
        (OUTPUT_EVAL,       "Per-pair evaluation scores"),
        (OUTPUT_METRICS,    "Full metrics report"),
    ]:
        exists = "✓" if os.path.exists(fname) else "✗"
        print(f"  {exists}  {fname:<42} {desc}")

    print()
    ans = input("  Would you like to look up more synonyms now? (yes/no): ").strip().lower()
    if ans in ("yes", "y"):
        interactive_synonym_lookup(model)

    print("\n  Done!\n")

WordNet loaded successfully.
Bilingual dictionary loaded : 7693 entries  (isizulu_english_dict.json)

STEP 1: Loading corpus
Sentences loaded : 315465
Total tokens     : 9728219

STEP 2: Training Word2Vec model
Vocabulary size  : 7995
NOTE: Word2Vec vocabulary is fixed to words seen during training.
      Any word not in this vocab cannot be embedded at inference time.


  INTERACTIVE SYNONYM LOOKUP
  Enter an isiZulu word to retrieve its synonyms.
  Candidates are filtered by matching noun class.
  NOTE: Word2Vec cannot look up out-of-vocabulary words.
  Commands:
    topn N   — change how many results to show (default 10)
    quit     — exit lookup mode


  Detected noun class group : 0  (prefixes: aba, abe, ob, um, umu)

  Word2Vec synonym candidates for 'umzali'
  (noun-class filtered — 3 candidates removed):
  #    Word                        Cosine  Label
  -------------------------------------------------------
  1    wengane                     0.8591  Strong
  2    uthisha    